In [37]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT_DIR = Path().absolute().parent.parent.parent.parent
PROJECT_ROOT_DIR

if str(PROJECT_ROOT_DIR.absolute()) not in sys.path:
    sys.path.append(str(PROJECT_ROOT_DIR.absolute()))

from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecCheckNan

from runF16.algorithm.models.sb3_model import PPOWithBCLoss
from runF16.algorithm.utils.sb3_eval_callback import MyEvalCallbackSTAT
from runF16.algorithm.utils.sb3_evaluate_policy import evaluate_policy_with_stat
from runF16.env.utils.sb3_utils import get_vec_env_dense_reward
from runF16.algorithm.utils.sb3_schedule import linear_schedule



In [38]:
RL_MODEL = "10hz_128_128_2e8steps_loss_2"

In [39]:
ROLLOUT_PROCESS_NUM = 10
STEP_FREQUENCE = 10
MAX_SIMULATE_TIME = 40
GAMMA = 0.995
REWARD_SCALAR_B = 0.5
EXPERT_DATA_DIR = "10hz_10_5_5"
RL_LR_RATE = 3e-4
KL_WITH_BC_MODEL_COEF = 1e-3

In [40]:
vec_env = VecCheckNan(get_vec_env_dense_reward(**{
    "num_process": ROLLOUT_PROCESS_NUM, 
    "logger": None, 
    "step_frequence": STEP_FREQUENCE,
    "max_simulate_time": MAX_SIMULATE_TIME,
    "gamma": GAMMA,
    "reward_scale_factor_b": REWARD_SCALAR_B,  # GuideEnvDenseReward中使用，取值小于1效果相当于在数值上放大了奖励，大于1相当于缩小了奖励
    "available_targets_dir": PROJECT_ROOT_DIR / "my_logs_for_parallel" / EXPERT_DATA_DIR,
    "sample_target_noise_std": [0., 0., 0.],
}))
eval_callback_env = VecCheckNan(get_vec_env_dense_reward(**{
    "num_process": ROLLOUT_PROCESS_NUM, 
    "logger": None, 
    "step_frequence": STEP_FREQUENCE,
    "max_simulate_time": MAX_SIMULATE_TIME,
    "gamma": GAMMA,
    "reward_scale_factor_b": REWARD_SCALAR_B,  # GuideEnvDenseReward中使用，取值小于1效果相当于在数值上放大了奖励，大于1相当于缩小了奖励
    "available_targets_dir": PROJECT_ROOT_DIR / "my_logs_for_parallel" / EXPERT_DATA_DIR,
    "sample_target_noise_std": [0., 0., 0.],
}))

In [41]:
rl_policy_save_dir = PROJECT_ROOT_DIR / "runF16" / "algorithm" / "checkpoints_sb3" / "rl" / RL_MODEL
algo_ppo_for_kl_loss = PPOWithBCLoss.load(str((rl_policy_save_dir / "best_model").absolute()))
algo_ppo_for_kl_loss.policy.set_training_mode(False)
algo_ppo = PPOWithBCLoss.load(
    str((rl_policy_save_dir / "best_model").absolute()), 
    env=vec_env, 
    seed=1,
    custom_objects={
        "bc_trained_algo": algo_ppo_for_kl_loss,
        "learning_rate": linear_schedule(RL_LR_RATE),
    },
    kl_coef_with_bc=KL_WITH_BC_MODEL_COEF,
)

In [42]:
avg_reward, avg_length, success_rate, infos = evaluate_policy_with_stat(
    model=algo_ppo.policy,
    env=vec_env,
    n_eval_episodes=10,
    deterministic=True,
)

/home/ucav/PythonProjects/618/runF16/algorithm/utils/sb3_evaluate_policy.py:215: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 51。target: (170.0, 25.0, 10.0)。expert steps: 116。
print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 61。target: (230.0, -5.0, -25.0)。expert steps: 150。
print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 68。target: (160.0, 10.0, -30.0)。expert steps: 115。
print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 78。target: (160.0, 10.0, 45.0)。expert steps: 128。
print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 117。target: (230.0, 10.0, 100.0)。expert steps: 167。
print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 119。target: (240.0, -25.0, 130.0)。expert steps: 326。
print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 138。target: (190.0, 5.0, 155.0)。expert steps: 299。
print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 148。target: (250.0, -10.0, 140.0)。expert s

In [43]:
len(infos)
for info in infos:
    tmp_env_info = info["last_info"]
    print(tmp_env_info["target_v"], tmp_env_info["target_mu"], tmp_env_info["target_chi"], tmp_env_info["is_success"], type(tmp_env_info["is_success"]))

170.0 25.0 10.0 True <class 'bool'>
230.0 -5.0 -25.0 True <class 'bool'>
160.0 10.0 -30.0 True <class 'bool'>
160.0 10.0 45.0 True <class 'bool'>
230.0 10.0 100.0 True <class 'bool'>
240.0 -25.0 130.0 True <class 'bool'>
190.0 5.0 155.0 True <class 'bool'>
250.0 -10.0 140.0 True <class 'bool'>
290.0 -5.0 -135.0 True <class 'bool'>
160.0 -5.0 10.0 True <class 'bool'>
